# 08. 종합 프로젝트: 로컬 Triage 수집기


## Goal

허가된 로컬 시스템에서 읽기 전용 정보를 수집하고, 실행 로그와 체크섬을 남기는 작은 triage 도구를 완성합니다. 이 실습은 외부 호스트에 접속하거나 스캔하지 않습니다.


## Setup


In [ ]:
from pathlib import Path
import os
import shutil
import tempfile

lab_dir = Path(tempfile.mkdtemp(prefix="bash-book-08-"))
os.environ["BASH_LAB_DIR"] = str(lab_dir)
print(f"격리된 실습 디렉터리: {lab_dir}")


## Steps

### 1. 수집기 작성


In [ ]:
%%bash
set -euo pipefail
cat > "$BASH_LAB_DIR/collect.sh" <<'BASH'
#!/usr/bin/env bash
set -euo pipefail
umask 077

if [[ $# -ne 1 ]]; then
  printf 'Usage: %s <output-directory>\n' "$0" >&2
  exit 64
fi

output=$1
mkdir -p -- "$output"
log="$output/collection.log"
report="$output/system-report.txt"

log_message() { printf '%s %s\n' "$(date -u '+%Y-%m-%dT%H:%M:%SZ')" "$*" | tee -a "$log"; }

log_message 'collection started'
{
  printf '== host ==\n'
  hostname
  printf '\n== kernel ==\n'
  uname -a
  printf '\n== disk ==\n'
  df -h .
  printf '\n== processes ==\n'
  ps -eo pid,ppid,user,comm | sed -n '1,21p'
} > "$report"

if command -v shasum >/dev/null 2>&1; then
  shasum -a 256 "$report" > "$output/SHA256SUMS"
else
  sha256sum "$report" > "$output/SHA256SUMS"
fi
log_message 'collection completed'
BASH
chmod 700 "$BASH_LAB_DIR/collect.sh"


### 2. 수집 실행


In [ ]:
%%bash
set -euo pipefail
output="$BASH_LAB_DIR/output"
"$BASH_LAB_DIR/collect.sh" "$output"
find "$output" -maxdepth 1 -type f -print | sort
printf '%s\n' '--- report preview ---'
sed -n '1,18p' "$output/system-report.txt"


### 3. 결과 무결성 검증


In [ ]:
%%bash
set -euo pipefail
cd "$BASH_LAB_DIR/output"
if command -v shasum >/dev/null 2>&1; then
  shasum -a 256 -c SHA256SUMS
else
  sha256sum -c SHA256SUMS
fi
grep -q 'collection started' collection.log
grep -q 'collection completed' collection.log
test -s system-report.txt
printf 'all capstone checks passed\n' ''


## Checks

- 출력 디렉터리에 보고서, 실행 로그, 체크섬이 생성되는가?
- 보고서가 비어 있지 않고 체크섬 검증이 성공하는가?
- 인자가 없을 때 사용법을 출력하고 실패하는가?
- 파일 권한과 허가된 실행 범위를 설명할 수 있는가?


## Next Steps

1. 수집 항목별 함수를 분리합니다.
2. `--output`, `--help` 옵션을 추가합니다.
3. Linux 환경에서 ShellCheck와 Bats 테스트를 추가합니다.
4. 결과 보존 기간과 민감정보 취급 절차를 문서화합니다.


In [ ]:
import shutil
from pathlib import Path
import os

lab_dir = Path(os.environ["BASH_LAB_DIR"])
shutil.rmtree(lab_dir, ignore_errors=True)
print(f"정리 완료: {lab_dir}")
